# 02. Prompting Baselines — CLadder · Corr2Cause

Before training Humm-LM and Humm-Fast, we establish baselines for how a general-purpose model scores on causal reasoning benchmarks. We measure two approaches together.

| Approach | Description | Corresponding module |
|---|---|---|
| **Noul** | Reads only the probability of the answer token (Yes/No) without generating. Along with accuracy, we look at calibration (Brier, ECE) | Humm-Fast |
| **CoT** | Generates a solution with the six CausalCoT steps and parses the final answer | Humm-LM |

- Hardware: one GPU per model (`cuda:0`, `cuda:1`).
- CLadder uses 200 items per Rung (association, intervention, counterfactual). CoT is slow, so it uses only 50 items per Rung. The solutions are long and 30-40% were truncated at 1024 tokens, so we use 4096 tokens.
- For Corr2Cause we look at both the original test set and the variable-renaming perturbation (refactorization) test set. A large performance drop on the perturbed set means the model memorized surface patterns.

In [ ]:
import gc

import pandas as pd
import torch

from humm.bench import datasets as D, prompts as P, runner as R
from humm.providers.hf import HFProvider

torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]

## 1. Data

In [ ]:
cladder = D.load_cladder(n_per_rung=200, seed=0)
cladder_cot = cladder.groupby("rung").head(50)
c2c = D.load_corr2cause("default")
c2c_ref = D.load_corr2cause("refactorization", n=1000, seed=0)

pd.DataFrame({"n": [len(cladder), len(cladder_cot), len(c2c), len(c2c_ref)],
              "pos_rate": [cladder.label.mean(), cladder_cot.label.mean(), c2c.label.mean(), c2c_ref.label.mean()]},
             index=["cladder", "cladder_cot", "corr2cause", "corr2cause_ref"]).round(3)

In [ ]:
print(cladder.prompt.iloc[0])
print("→ label:", cladder.label.iloc[0], "| rung:", cladder.rung.iloc[0], "| query:", cladder.query_type.iloc[0])

## 2. Models

We fix two models and load one per GPU.

| Model | GPU | Role |
|---|---|---|
| Qwen/Qwen3.5-4B | `cuda:0` | Basis for Humm-Fast, lightweight comparison |
| Qwen/Qwen3.5-9B | `cuda:1` | Main model for Humm-LM |

Qwen3.5 requires transformers 5.2 or later. Thinking mode is the default, but `HFProvider` calls with `enable_thinking=False`, so Noul reads the answer token directly and CoT writes the solution as the CausalCoT prompt instructs.

In [ ]:
MODELS = {
    "Qwen/Qwen3.5-4B": "cuda:0",
    "Qwen/Qwen3.5-9B": "cuda:1",
}
BATCH = 16

In [ ]:
# Qwen3.5 has linear attention layers, so first check that left-padded batch results match single-item results
lm = HFProvider("Qwen/Qwen3.5-4B", device="cuda:0", batch_size=BATCH)
probe = [P.cladder_noul(r) for _, r in cladder.head(8).iterrows()]
batched = [p[0] for p in lm.option_probs(probe, R.YES_NO)]
single = [lm.option_probs([x], R.YES_NO)[0][0] for x in probe]
print(pd.DataFrame({"batched": batched, "single": single}).round(4))
del lm
gc.collect()
torch.cuda.empty_cache()

## 3. Running

We load one model, run all four evaluations, free the memory, and move on to the next model. Results are saved per item to `results/baseline_qwen35.csv` at the repository root (regardless of the notebook's location).

In [ ]:
results = []
for model_id, device in MODELS.items():
    lm = HFProvider(model_id, device=device, batch_size=BATCH)
    results += [
        R.run_noul(lm, cladder, P.cladder_noul).assign(bench="cladder"),
        R.run_noul(lm, c2c, P.corr2cause_noul).assign(bench="corr2cause"),
        R.run_noul(lm, c2c_ref, P.corr2cause_noul).assign(bench="corr2cause_ref"),
        R.run_cot(lm, cladder_cot, P.cladder_cot, max_new_tokens=4096).assign(bench="cladder"),
    ]
    del lm
    gc.collect()
    torch.cuda.empty_cache()

res = pd.concat(results, ignore_index=True)
R.save(res, "baseline_qwen35")

## 4. Results

We look at the per-benchmark summary and the CLadder summary by Rung. As prior research suggests, accuracy should drop on Rungs 2 and 3 (intervention, counterfactual), and Corr2Cause should be close to chance.

In [ ]:
R.summarize(res, by=["bench"]).round(3)

In [ ]:
R.summarize(res[res.bench == "cladder"], by=["rung"]).round(3)

In [ ]:
R.summarize(res[res.bench == "cladder"], by=["query_type"]).pivot_table(index="query_type", columns=["model", "mode"], values="acc").round(3)

## 5. Inspecting CoT output

We read the solutions of items that failed to parse and items answered incorrectly. Whether errors arise in graph extraction, in deriving the estimand, or in arithmetic informs the design of the Humm-Gym data.

In [ ]:
cot = res[res["mode"] == "cot"]
wrong = cot[cot.pred != cot.label]
print(len(wrong), "wrong /", len(cot))
print(wrong.iloc[0].prompt, "\n---\n", wrong.iloc[0].output[-1500:])

## 6. (Optional) Anthropic API baseline

We run the same CoT evaluation with Claude for comparison. The API does not return logits, so only the CoT approach is possible.

The API key is taken from the environment variable `ANTHROPIC_API_KEY` if set, otherwise it is requested in an input box. The key you enter does not remain on screen or in the notebook file.

In [ ]:
import os
from getpass import getpass

from humm.providers.anthropic_api import AnthropicProvider

os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
api = AnthropicProvider("claude-sonnet-5-5")
api.complete(["Reply with OK."], max_new_tokens=5)

In [ ]:
res_api = R.run_cot(api, cladder_cot, P.cladder_cot, max_new_tokens=4096).assign(bench="cladder")
R.save(res_api, "baseline_api")
R.summarize(res_api, by=["rung"]).round(3)

## Result files

Save an executed notebook under a different name than the original, and keep the original in `notebooks/` without outputs.

| File | Required | Notes |
|---|---|---|
| `results/baseline_qwen35.csv` | Required | Per-item Qwen3.5 Noul and CoT results |
| `results/baseline_api.csv` | If the API cell was run | Claude CoT results |
| `data/bench/` | Do not copy | Cache of the original benchmark data |